# site： https://medium.com/@me.mdhamim/a-comprehensive-guide-to-python-threading-advanced-concepts-and-best-practices-9f3aea6f0a63

In [1]:
# ex1
import threading
import time


# Function to be executed in the thread
def print_numbers():
    for i in range(5):
        time.sleep(1)  # Simulating some work
        print(f"Thread: {threading.current_thread().name}, Number: {i}")
# Create a thread
my_thread = threading.Thread(target=print_numbers)
# Start the thread
my_thread.start()
# Main thread continues to execute independently
for i in range(5):
    time.sleep(1)
    print(f"Main Thread, Number: {i}")

Main Thread, Number: 0Thread: Thread-3 (print_numbers), Number: 0

Main Thread, Number: 1
Thread: Thread-3 (print_numbers), Number: 1
Thread: Thread-3 (print_numbers), Number: 2Main Thread, Number: 2

Thread: Thread-3 (print_numbers), Number: 3
Main Thread, Number: 3
Main Thread, Number: 4
Thread: Thread-3 (print_numbers), Number: 4


In [6]:
# ex2 the Lock
from threading import Thread,Lock

share_res = 0
lock = Lock()
def deal_with_share_res():
    global share_res
    for i in range(1000000):
        with lock:
            share_res += 1

t1 = Thread(target=deal_with_share_res)
t2 = Thread(target=deal_with_share_res)

t1.start()
t2.start()
t1.join()
t2.join()
print(f"Final value of shared resource: {share_res}")


Final value of shared resource: 2000000


In [7]:
# ex3 the SemaPhore
from threading import Thread,Semaphore

share_res = 0
semaphore = Semaphore()
def deal_with_share_res():
    global share_res
    for i in range(1000000):
        with semaphore:
            share_res += 1

t1 = Thread(target=deal_with_share_res)
t2 = Thread(target=deal_with_share_res)

t1.start()
t2.start()
t1.join()
t2.join()
print(f"Final value of shared resource: {share_res}")



Final value of shared resource: 2000000


In [11]:
# ex4  Using Queues for Thread Communication
from threading import Thread
from queue import Queue
import time

msg_que = Queue()
def produce():
    for i in range(5):
        time.sleep(1)
        msg_que.put(f"Message {i}")

def consume():
    while True:
        msg = msg_que.get()
        if msg == "Stop":
            break
        print(f"Cousuming message:{msg}")    
        
        
t_prod = Thread(target=produce)
t_cons = Thread(target=consume)

t_prod.start()
t_cons.start()

t_prod.join()
msg_que.put("Stop")              
t_cons.join()        

Cousuming message:Message 0
Cousuming message:Message 1
Cousuming message:Message 2
Cousuming message:Message 3
Cousuming message:Message 4


In [13]:
# ex5 Event Objects for Signaling
from threading import Thread,Event

event = Event()
def wait_for_event():
    print("Waiting for event...")
    event.wait()
    print("Event has been set...")

def set_event():
    print("Setting Event...")
    print("Event Set...")
    event.set()

t1= Thread(target=wait_for_event)
t2 = Thread(target=set_event)

t1.start()
t2.start()

t1.join()           
t2.join()  
    

Waiting for event...
Setting Event...
Event Set...
Event has been set...


In [16]:
# ex6
from threading import Thread

# Immutable object (tuple) example
immutable_data = (11,22,33)

def process_immutable_data(data):
    total = sum(data)
    print(f"Result is :{total}")

t1= Thread(target=process_immutable_data,args=(immutable_data,))
t2 = Thread(target=process_immutable_data,args=(immutable_data,))

t1.start()
t2.start()

t1.join()           
t2.join()  



Result is :66
Result is :66


In [17]:
# ex7
from threading import Thread
import time

def deamon_task():
    print("Deamon Thread is running...")
    time.sleep(1)

deamon_thread = Thread(target=deamon_task)
deamon_thread.deamon = True
deamon_thread.start()
time.sleep(5)
print("Main thread is done.")
    
    


Deamon Thread is running...
Main thread is done.


In [19]:
# ex8 ThreadPoolExecutor and Concurrent Futures
from concurrent.futures import ThreadPoolExecutor
import concurrent
import time

def time_consuming_task(task_id):
    print(f"Task {task_id} started...")
    time.sleep(2)
    print(f"Task {task_id} completed...")
    return f"Result from Task {task_id}"

with ThreadPoolExecutor(max_workers=3) as executor:
    future1 = executor.submit(time_consuming_task,1)
    future2 = executor.submit(time_consuming_task,2)
    future3 = executor.submit(time_consuming_task,3)
    # Gather results
    results = [future.result() for future in concurrent.futures.as_completed([future1,future2,future3])]

print("All tasks completed.")
print("Results:", results)    
    


Task 1 started...
Task 2 started...
Task 3 started...
Task 1 completed...
Task 2 completed...
Task 3 completed...
All tasks completed.
Results: ['Result from Task 1', 'Result from Task 2', 'Result from Task 3']


In [2]:
# ex9 Multithreading vs. Multiprocessing
import multiprocessing
import concurrent

# Function to simulate a CPU-bound task
def cpu_bound_task(task_id):
    result = 0
    for _ in range(10**7):
        result += 1
    print(f"CPU-bound Task {task_id} completed.")
    return result
# Using multithreading
with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
    futures_threading = [executor.submit(cpu_bound_task, i) for i in range(2)]
    results_threading = [future.result() for future in concurrent.futures.as_completed(futures_threading)]
# Using multiprocessing
with concurrent.futures.ProcessPoolExecutor(max_workers=2) as executor:
    futures_multiprocessing = [executor.submit(cpu_bound_task, i) for i in range(2)]
    results_multiprocessing = [future.result() for future in concurrent.futures.as_completed(futures_multiprocessing)]
print("Multithreading results:", results_threading)
print("Multiprocessing results:", results_multiprocessing)  
    


CPU-bound Task 0 completed.
CPU-bound Task 1 completed.


Traceback (most recent call last):
  File "D:\Programs\Python311\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "D:\Programs\Python311\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "D:\Programs\Python311\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is closed


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [29]:
# ex10 8.1 Race Conditions
import threading


shared_variable = 0
lock = threading.Lock()
def modify_shared_variable():
    global shared_variable
    with lock:
        shared_variable += 1
# Create and start multiple threads
threads = [threading.Thread(target=modify_shared_variable) for _ in range(10)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("Final value of shared variable:", shared_variable)


Final value of shared variable: 10


In [ ]:
# ex11 8.2 Priority Inversion
# ex12 8.3 Overhead and Scalability Concerns
import concurrent.futures


def task():
    # Some computation or I/O operation
    pass
# Using ThreadPoolExecutor to manage threads
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    results = list(executor.map(task, range(10)))


In [30]:
# ex13 8.4 Lack of Thread Safety: Accessing shared data without proper synchronization can lead to data corruption and unexpected behavior.
import threading


shared_data = []
def modify_shared_data(item):
    global shared_data
    with threading.Lock():
        shared_data.append(item)
# Create and start multiple threads
threads = [threading.Thread(target=modify_shared_data, args=(i,)) for i in range(10)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print("Final shared data:", shared_data)

Final shared data: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [7]:
# 9.1 Web Scraping with Concurrent Requests
import threading
import requests

def fetch_data(url):
    data = requests.get(url)
    print(f"Fetched data from {url}")
# List of URLs to scrape
urls = ["https://example.com", "https://example.org", "https://example.net"]
threads = [threading.Thread(target=fetch_data,args=(url,)) for url in urls]

for i in range(len(threads)):
    threads[i].start()
    threads[i].join()
print("Web scraping completed.")    

Fetched data from https://example.com
Fetched data from https://example.org
Fetched data from https://example.net
Web scraping completed.


In [9]:
from PIL import Image,ImageFilter
from io import BytesIO
import requests
from concurrent.futures import ProcessPoolExecutor

def download_image(url):
    response = requests.get(url)
    return Image.open(BytesIO(response.content))

def apply_effect(image):
    # Simulate a CPU-intensive task by applying a Gaussian blur effect
    return image.filter(ImageFilter.GaussianBlur(radius=2))

def process_image(image_url):
    image = download_image(image_url)
    image2 = apply_effect(image)
    print(f"Processed image with size {image2.size}")
    image.save(f"processed_{image_url.split('/')[-1]}.png")
    return image2
if __name__ == "__main__":
    image_urls = [
    "https://www.val-gardena.com/fileadmin/_processed_/8/5/csm_2020-09-09-jb-pic_c31fd6a4c2.jpg",
    "https://static.vecteezy.com/system/resources/thumbnails/036/333/113/small_2x/monarch-beautiful-butterflygraphy-beautiful-butterfly-on-flower-macrography-beautyful-nature-photo.jpg",
    "https://static.vecteezy.com/system/resources/thumbnails/071/827/814/small_2x/toyota-4runner-exploring-the-amazon-rainforests-lush-trails-free-photo.jpg",
    # Add more URLs as needed
    ]

    with ProcessPoolExecutor() as executor:
        processed_images = list(executor.map(process_image, image_urls))
    print("Image processing completed.")

Traceback (most recent call last):
  File "D:\Programs\Python311\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "D:\Programs\Python311\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "D:\Programs\Python311\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is closed
Traceback (most recent call last):
  File "D:\Programs\Python311\Lib\multiprocessing\queues.py", line 246, in _feed
    send_bytes(obj)
  File "D:\Programs\Python311\Lib\multiprocessing\connection.py", line 184, in send_bytes
    self._check_closed()
  File "D:\Programs\Python311\Lib\multiprocessing\connection.py", line 137, in _check_closed
    raise OSError("handle is closed")
OSError: handle is closed


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.